1. 共用參數設定


In [ ]:
'''
Measure the elapsed time of every code cell, including cells that raise errors.
記錄每個 Code Cell 的執行時間，即使發生錯誤也顯示已耗費的時間。
'''
from time import perf_counter as _cell_perf_counter
from IPython import get_ipython as _cell_get_ipython

_CELL_TIMER_SHELL = _cell_get_ipython()
if _CELL_TIMER_SHELL is None:
    raise RuntimeError('請在 Jupyter Notebook 中執行本教材。')
_CELL_TIMER_STATE = {'started': _cell_perf_counter()}

def _cell_timer_begin(_info):
    _CELL_TIMER_STATE['started'] = _cell_perf_counter()

def _cell_timer_end(result):
    started = _CELL_TIMER_STATE.pop('started', None)
    if started is None:
        return
    minutes, seconds = divmod(_cell_perf_counter() - started, 60)
    elapsed = f'{int(minutes)} 分 {seconds:05.2f} 秒' if minutes else f'{seconds:.2f} 秒'
    status = '執行失敗' if result.error_before_exec or result.error_in_exec else '執行完成'
    print(f'⏱ 本格{status}，耗時：{elapsed}')

'''
Replace existing timers when the first cell is executed again.
重新執行第一格時，先取消舊計時器，避免重複顯示耗時。
'''
for _event_name, _callback in globals().get('_CELL_TIMER_CALLBACKS', ()):
    try:
        _CELL_TIMER_SHELL.events.unregister(_event_name, _callback)
    except ValueError:
        pass
_CELL_TIMER_CALLBACKS = (
    ('pre_run_cell', _cell_timer_begin),
    ('post_run_cell', _cell_timer_end),
)
for _event_name, _callback in _CELL_TIMER_CALLBACKS:
    _CELL_TIMER_SHELL.events.register(_event_name, _callback)

'''
Import standard-library tools for editable paths and local service settings.
匯入設定路徑與本機服務參數所需的 Python 內建工具。
'''
import os
from pathlib import Path
ROOT = Path('/kaggle/working')
COMFY = ROOT / 'ComfyUI-ZImageTurbo'

'''
Select the pinned ComfyUI release; update only after verifying custom nodes.
選擇固定的 ComfyUI 版本，更新前須先驗證自訂節點相容性。
'''
COMFY_REF = 'v0.45.0'
COMFY_PORT = 8189  # 8189、8190 等未占用連接埠
HF_CACHE_DIR = Path('/tmp/zimage_hf')

'''
Apply the cache path before model downloads begin.
模型下載之前先套用設定的快取路徑。
'''
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')
REMOTE_PORT = 8765  # 例如 8765、8766，範圍 1024～65535
REMOTE_SERVICE_ID = '06_Z-Image-Turbo_圖片生成_Kaggle'

'''
Place optional API job data inside the configured workspace.
將遠端工作與暫存資料存放在第一格設定的工作空間底下。
'''
REMOTE_ROOT = ROOT / 'remote_api_jobs' / REMOTE_SERVICE_ID
REMOTE_QUEUE_CAPACITY = 32  # 建議 12～32，依課堂人數調整
REMOTE_UPLOAD_CONCURRENCY = 2  # 建議 1～2

'''
Choose output retention and storage safety limits.
設定遠端成果保留期限、空間限制及主機磁碟最低剩餘量。
'''
REMOTE_RESULT_TTL_HOURS = 24
REMOTE_RESULT_LIMIT_GIB = 8
REMOTE_MIN_FREE_GIB = 2
REMOTE_STALL_HOURS = 2
assert 1024 <= REMOTE_PORT <= 65535, 'REMOTE_PORT 必須介於 1024～65535。'
assert 1 <= REMOTE_QUEUE_CAPACITY <= 1024, 'FIFO 佇列容量設定無效。'
assert 1 <= REMOTE_UPLOAD_CONCURRENCY <= 8, '上傳並發限制設定無效。'
assert REMOTE_RESULT_TTL_HOURS > 0 and REMOTE_RESULT_LIMIT_GIB > 0 and REMOTE_MIN_FREE_GIB >= 0
assert REMOTE_STALL_HOURS > 0

'''
Avoid a local ComfyUI port collision with the optional FIFO API.
避免 ComfyUI 與選用 FIFO API 使用同一個本機 Port。
'''
assert 1024 <= COMFY_PORT <= 65535 and COMFY_PORT != REMOTE_PORT
print('✓ 第 1 節完成：共用參數已設定，請繼續執行環境安裝。')


2. 安裝套件


In [ ]:
'''
Import the tools for installing ComfyUI.
匯入安裝 ComfyUI 所需工具。
'''
import subprocess
import sys
from pathlib import Path

REPOS = [
    ('https://github.com/Comfy-Org/ComfyUI.git', COMFY),
    ('https://github.com/city96/ComfyUI-GGUF.git', COMFY / 'custom_nodes/ComfyUI-GGUF'),
    ('https://github.com/pollockjj/ComfyUI-MultiGPU.git', COMFY / 'custom_nodes/ComfyUI-MultiGPU'),
]
old_commits = {}
for url, folder in REPOS:
    if folder.exists() and not (folder / '.git').exists():
        raise RuntimeError(f'不是 Git 倉庫：{folder}')
    if (folder / '.git').exists():
        old_commits[folder] = subprocess.check_output(['git', '-C', str(folder), 'rev-parse', 'HEAD'], text=True).strip()
    else:
        command = ['git', 'clone', '-q', '--depth', '1']

        '''
        Install ComfyUI at the verified official release tag.
        安裝 ComfyUI 時指定官方穩定版標籤。
        '''
        if folder == COMFY:
            command += ['--branch', COMFY_REF]
        subprocess.run(command + [url, str(folder)], check=True)
    if folder.name == 'ComfyUI-MultiGPU':
        print('MultiGPU v2.6.4 已停止維護，與新版 ComfyUI 的相容性仍須在 GPU 上驗證。')

    elif folder != COMFY:
        '''
        Keep user modifications intact rather than force-resetting them.
        保留使用者修改，不進行強制重設。
        '''
        if not subprocess.check_output(['git', '-C', str(folder), 'status', '--porcelain'], text=True).strip():
            subprocess.run(['git', '-C', str(folder), 'pull', '--ff-only', '-q'], check=True)
        else:
            print(f'略過已有修改的外掛：{folder.name}')

'''
Retrieve the official ComfyUI release even for an existing checkout.
現有 ComfyUI 倉庫也要取得指定的正式發行標籤。
'''
subprocess.run(['git', '-C', str(COMFY), 'fetch', '--depth', '1', 'origin', 'tag', COMFY_REF], check=True)
subprocess.run(['git', '-C', str(COMFY), 'checkout', '--quiet', '--detach', COMFY_REF], check=True)
ZIMAGE_CODE_UPDATED = any(
    folder in old_commits and old_commits[folder] != subprocess.check_output(
        ['git', '-C', str(folder), 'rev-parse', 'HEAD'], text=True,
    ).strip()
    for _, folder in REPOS
)

'''
Report all active upstream revisions.
顯示每個原始碼倉庫的實際版本。
'''
for _, folder in REPOS:
    print(folder.name, subprocess.check_output(['git', '-C', str(folder), 'rev-parse', '--short', 'HEAD'], text=True).strip())

installed = ROOT / '.zimage_kaggle_v045_installed'
import torch

'''
Store the installed PyTorch version as a pip constraint.
將既有 PyTorch 版本設為 pip 限制。
'''
constraint = ROOT / 'zimage_torch_constraint.txt'
constraint.write_text(f'torch=={torch.__version__}\n', encoding='utf-8')
if not installed.exists():
    '''
    Install the official ComfyUI dependencies.
    安裝 ComfyUI 官方依賴套件。
    '''
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), '-r', str(COMFY / 'requirements.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), '-r', str(COMFY / 'custom_nodes/ComfyUI-GGUF/requirements.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), 'huggingface_hub', 'hf_xet', 'requests'], check=True)
    installed.touch()

'''
Confirm that installation is ready.
顯示安裝完成狀態。
'''
print('ComfyUI installed')
print('✓ 第 2 節完成：套件與圖片生成環境已準備完成。')


3. 下載模型


In [ ]:
'''
Import the official Hugging Face downloader.
匯入 Hugging Face 官方下載工具。
'''
from huggingface_hub import hf_hub_download
MODEL_FILES = [
    ('unsloth/Z-Image-Turbo-GGUF', 'z-image-turbo-Q6_K.gguf', 'unet/z-image-turbo-Q6_K.gguf'),
    ('Comfy-Org/z_image_turbo', 'split_files/text_encoders/qwen_3_4b_fp8_mixed.safetensors', 'text_encoders/qwen_3_4b_fp8_mixed.safetensors'),
    ('Comfy-Org/z_image_turbo', 'split_files/vae/ae.safetensors', 'vae/ae.safetensors'),
]
for repo, name, relative_path in MODEL_FILES:
    destination = COMFY / 'models' / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.is_file():
        if destination.is_symlink():
            destination.unlink()
        cached_file = hf_hub_download(repo_id=repo, filename=name)
        destination.symlink_to(cached_file)
    print('Ready:', destination.name)

print('✓ 第 3 節完成：模型權重已下載並準備完成。')


4. 啟動模型服務


In [ ]:
'''
Import the tools for managing a persistent local server.
匯入管理常駐服務所需工具。
'''
import time
import requests
import torch
assert torch.cuda.device_count() >= 2, '請在 Kaggle 設定中選擇 GPU T4 ×2。'
BASE = f'http://127.0.0.1:{COMFY_PORT}'
LOG_PATH = ROOT / 'zimage_server.log'

'''
Check whether the model server is already running.
檢查模型服務是否已經啟動。
'''
try:
    running = requests.get(BASE + '/system_stats', timeout=3).ok

except requests.RequestException:
    running = False

if running and globals().get('ZIMAGE_CODE_UPDATED', False):
    '''
    Require an explicit restart for processes not owned by this setup cell.
    服務可能不由此 Cell 管理，因此要求先關閉舊行程。
    '''
    raise RuntimeError(f'ComfyUI 程式碼已更新；請先停止 {COMFY_PORT} 連接埠的舊服務，再執行本 Cell。')

if not running:
    log_file = LOG_PATH.open('w')
    server = subprocess.Popen(
        [sys.executable, str(COMFY / 'main.py'), '--listen', '127.0.0.1',
         '--port', str(COMFY_PORT), '--highvram', '--cache-classic', '--preview-method', 'none'],
        cwd=str(COMFY), stdout=log_file, stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    log_file.close()

'''
Wait for ComfyUI to complete startup and register custom nodes.
等待 ComfyUI 啟動完成並註冊自訂節點。
'''
for attempt in range(120):
    '''
    Check whether the server responds.
    檢查服務是否能正常回應。
    '''
    try:
        if requests.get(BASE + '/system_stats', timeout=3).ok:
            break

    except requests.RequestException:
        pass
    time.sleep(2)

else:
    raise RuntimeError(f'ComfyUI 啟動失敗，請檢查 {LOG_PATH}')

node_info = requests.get(BASE + '/object_info', timeout=60).json()
required_nodes = {'UnetLoaderGGUFMultiGPU', 'CLIPLoaderMultiGPU', 'VAELoaderMultiGPU'}

'''
Detect missing custom nodes before submitting a workflow.
在生成前先檢查是否缺少自訂節點。
'''
missing_nodes = required_nodes - node_info.keys()
assert not missing_nodes, f'找不到雙 GPU 節點：{sorted(missing_nodes)}；請檢查 {LOG_PATH}'
print('GPU 0:', torch.cuda.get_device_name(0), '｜影像生成')
print('GPU 1:', torch.cuda.get_device_name(1), '｜文字編碼與 VAE')
print('ComfyUI ready:', BASE)
print('✓ 第 4 節完成：雙 GPU ComfyUI 服務已就緒。')


5. 生成設定


In [ ]:
'''
Describe the desired image.
設定圖片內容。
'''
PROMPT = '''

'''.strip()
WIDTH, HEIGHT, SEED = 640, 960, 42

'''
Preserve the original Turbo sampling parameters.
保留原本的 Turbo 採樣參數。
'''
STEPS, CFG = 8, 1
OUTPUT_NAME = 'zimage.png'
print('✓ 第 5 節完成：圖片參數已設定（尚未生成）。')


6. 生成、預覽與下載圖片


In [ ]:
'''
Serialize notebook generation with optional remote FIFO GPU jobs.
讓 Notebook 手動生成與選用的遠端 FIFO 共用 GPU 鎖，避免同時推論。
不必先啟動 ngrok 或任何 HTTP API。
'''
import threading
GENERATION_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
with GENERATION_LOCK:
    import uuid
    from IPython.display import FileLink, display
    from PIL import Image
    assert WIDTH > 0 and HEIGHT > 0 and WIDTH % 64 == 0 and HEIGHT % 64 == 0, '寬高須為正整數且為 64 的倍數。'
    import secrets
    RUN_SEED = secrets.randbelow(2**32) if SEED == -1 else int(SEED)

    '''
    Require a supported seed before building the inference workflow.
    在建立推論工作流程前，先檢查種子範圍。
    '''
    assert 0 <= RUN_SEED < 2**32, 'SEED 必須介於 0 至 4294967295，或設為 -1。'
    print(f'本次生成 seed：{RUN_SEED}')
    workflow = {
        '1': {'class_type': 'UnetLoaderGGUFMultiGPU', 'inputs': {'unet_name': 'z-image-turbo-Q6_K.gguf', 'device': 'cuda:0'}},
        '2': {'class_type': 'CLIPLoaderMultiGPU', 'inputs': {'clip_name': 'qwen_3_4b_fp8_mixed.safetensors', 'type': 'lumina2', 'device': 'cuda:1'}},
        '3': {'class_type': 'VAELoaderMultiGPU', 'inputs': {'vae_name': 'ae.safetensors', 'device': 'cuda:1'}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': PROMPT, 'clip': ['2', 0]}},
        '5': {'class_type': 'ConditioningZeroOut', 'inputs': {'conditioning': ['4', 0]}},
        '6': {'class_type': 'EmptySD3LatentImage', 'inputs': {'width': WIDTH, 'height': HEIGHT, 'batch_size': 1}},
        '7': {'class_type': 'ModelSamplingAuraFlow', 'inputs': {'shift': 3, 'model': ['1', 0]}},
        '8': {'class_type': 'KSampler', 'inputs': {'seed': RUN_SEED, 'steps': STEPS, 'cfg': CFG, 'sampler_name': 'res_multistep', 'scheduler': 'simple', 'denoise': 1, 'model': ['7', 0], 'positive': ['4', 0], 'negative': ['5', 0], 'latent_image': ['6', 0]}},
        '9': {'class_type': 'VAEDecode', 'inputs': {'samples': ['8', 0], 'vae': ['3', 0]}},
        '10': {'class_type': 'SaveImage', 'inputs': {'filename_prefix': 'zimage', 'images': ['9', 0]}},
    }
    response = requests.post(BASE + '/prompt', json={'prompt': workflow, 'client_id': str(uuid.uuid4())}, timeout=120)

    '''
    Surface validation errors instead of silently waiting.
    若工作流程驗證失敗，立即顯示錯誤。
    '''
    if not response.ok:
        raise RuntimeError(response.text)
    job_id = response.json()['prompt_id']
    start = time.monotonic()
    for attempt in range(1800):
        record = requests.get(BASE + '/history/' + job_id, timeout=30).json().get(job_id)
        if record:
            if record.get('status', {}).get('status_str') == 'error':
                raise RuntimeError(str(record['status']))
            images = record.get('outputs', {}).get('10', {}).get('images', [])
            if images:
                break

        '''
        Print periodic status in the cell output without assuming a step percentage.
        定期顯示已等待時間，不捏造採樣百分比。
        '''
        if attempt % 5 == 0:
            print(f'生成中… {time.monotonic() - start:.0f} 秒', flush=True)
        time.sleep(2)

    else:
        raise TimeoutError(f'生成逾時，請檢查 {LOG_PATH}')
    image_response = requests.get(BASE + '/view', params=images[0], timeout=120)

    '''
    Reject failed downloads to prevent corrupted files.
    防止下載失敗卻仍儲存不完整圖片。
    '''
    image_response.raise_for_status()
    filename = Path(OUTPUT_NAME).name
    if not filename.lower().endswith('.png'):
        filename += '.png'
    OUTPUT_PATH = ROOT / filename
    index = 2

    '''
    Preserve earlier generations by choosing an unused filename.
    選擇未使用的檔名，避免覆蓋先前作品。
    '''
    while OUTPUT_PATH.exists():
        '''
        Add a numeric suffix for each additional image.
        在重複檔名後加入編號。
        '''
        OUTPUT_PATH = ROOT / f'{Path(filename).stem}_{index:03d}.png'
        index += 1
    OUTPUT_PATH.write_bytes(image_response.content)
    print(f'完成：{OUTPUT_PATH.name}｜{time.monotonic() - start:.1f} 秒')
    display(Image.open(OUTPUT_PATH))
    display(FileLink(str(OUTPUT_PATH)))

assert OUTPUT_PATH.is_file() and OUTPUT_PATH.stat().st_size > 0, '找不到有效的 PNG。'
print('✓ 第 6 節完成：圖片已生成並顯示預覽與下載連結。')


7. （選用）啟動 ngrok 與 FIFO 遠端 API


In [ ]:
'''
Require the first configuration cell before enabling the optional remote API.
選用遠端 API 啟動前，必須先執行第一格共用參數設定。
'''
assert all(name in globals() for name in ('REMOTE_PORT', 'REMOTE_SERVICE_ID', 'REMOTE_ROOT')), '請先執行第 1 節共用參數設定。'
import zlib
import math
import re

import queue
import copy
import secrets
import io
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.request
import uuid
import zipfile
from email import policy
from email.parser import BytesParser
from getpass import getpass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import urlsplit, parse_qs

REMOTE_KIND = 'image_text'
REMOTE_FIELDS = ['task', 'prompt', 'width', 'height', 'steps', 'seed', 'cfg']
REMOTE_IS_LTX = False
REMOTE_SCRIPT = zlib.decompress(b'x\xda\x85Yks\xd3V\x1a\xfe\x9e_q\xbet\x1c\xb7\xe0\xe6\x02\x14\x98fg\xd8\x12B\xa6\\\xb2$\xec\xce,\xc3x\x14\xfb8V#K\xae.\ti&3\xa6CH\x82c\x92-$\xdcRB\xb8\xa4\x01\n)\x14\x86\x10B\xf8/\xbb\x96l\x7f\xea_\xd8\xe7=G\x92ecZ>\x10K::\xef\xfdy\x9f\xf7(\x16\x8b\xb5\xf5\xe7\xf2\x86i3\xdb04\x8be\x0c\x93\xd9\xa6\x92\x1aU\xf5\x11\xa6\xe8i\x96V\xad\xbc\xa6L\xd0\xa5\x9d\xe5l\x84\xeb\xdcTl\x9efjN\x19\xe1\x896w~\xd3\x9d~T\xfd\xf0\xbe\xfa\xf6a\xad\xf0\xd2\xdd^\xaf\xce\xce\xd4\xd66+\x0f\xb7\xdd\x95\xe5\xca\xdcL\xe5\xf6%\xf7\xcd#w\xfa\xcd\xff\n?\xb6\xc5 O\x95\xf2\x1cGM\xb7eL#\xc7\xfa\x07&\xec\xac\xa1\'|Q\xcc_pL\xd5\xf8\tU\x1f\xdd\x13\xe8 W\x0f\xf4\x9f\x08V\xf4\x93\nmb\xd3o\xb2<5*u\xc2\xf2\x1c\xd7-\xd5\xd0-6\xcca\x10g\x963\x9cSm[\xd8\xc4\xbe3\x86\x13m\xde\x93\xfb\xde\xea#\xa9\xa1\xfb\xdb\xb6\xbb\xb9U\xddx\xe6>\xbf\xed\xdd\xdct\x17\xd7\xbd\x959oi6PX\xb1,\x0ei\xff\xea?:t\x9c\xfd\x8du\x08\xbf\x1c\xef\xed\xef;>\x14^\xca\x87\x9f\xb1\x03\xfbXOO\xe3\x92\xf0\xe6\x1e\x16s7\x7f\xad=\xbdY\xbb7[\xf9q\xdb{\xf6\xc0[z\xe5-m\x95\xb7\xae\xe3\x92V\x91\xaf\n%\xdc\x82\xe8\x984\xac\xcfw8\x14\xcf\x98\xdc\xca\xb2\xee\xae\xbd\xc3\xaa\xcd,\x8e\x18\x18\xba6\xc1\xc6\xb3\\g{;\x99j1\x93\x7f\xefp\x0b\xd1A\\\x16\x9e\xc0\ni\x14=\xf5n\xfd\xe8\xcd\x95\xdcw\xdb\x95\xa7Eo\xf97H\nv\xaa\xdd\xde\xf0\x1e\xafV6\x9e\xbb\xcf\x16\x9bbd\xf1\x94\xc9mKjr\x86[\x866\xc6!\x94K\xe9yn2\xd3\xd1\xd9\xb8\x8a\xf0963\xc6\xb89n\xaav\x90+\xbaa\xf3a\xc3\x18\xc5j\xe1{x}s\xc1\xfbu\xadr}\xd5\x9b]\xf4^lC\xb5\xf2V\xc1-\x14\xa5\xf0?v\xe6\xcb\xef/\x97\xb7J\xd5\xf5\xcb\xee\xe6Sv*x\x9f\xdcru\xd5\xfd\xa5(\xcd\t\x94<s\xf6Tr\xb0\xb7\xf7(\xeb\t\x14M\x98\xf0\xfb0\xd7\x8c\xf1\xf6\xae\xcf?\xef\xee\x8a35\xc3\xe4\x92\x1er\x02\xd7,\xceT\xddn\xa7{\xf1\xc0\xac\xef\x1d\xd5$\xffZN\x9e\xac\x86e\xc2<?w\x86\x1dUK\x07&\xa9z\x86\x9b\\Oq6n\x98\xa3\x19\x08\x82\xa7W6|\xb7^\xdd\xa8>]F\xb2\x97\xdf\xafx\xaf/V6\x8a\xee\\\tF\xb9\xd3\xb32\xdd\xa4\x99\x95\xcdK\xeeJ\xa9)\xb9:\xd8\xd7=,4\xe8k&\xd4G\xc2\x88K\xf7\xc34R\xa6\xfc\x0e\x81\xdb\xc1\xca\xea\xcc\xefl_\xd7\xa1}\x87\x0e|\xd5uh?\x04x\xb3\xcb\xf0\x0c\xe5\xd0\xde\xcez\xe6\x0cf\x8dq\xa1\xb4\x92\xb2\x1dE\x93FY\x06\x0c%/\xa1\x92(v)E\x87\xa1H\x9c\xbci\xa4\x9d\x14VhH63\xd1\xe6\x97\xf0\xe6z\xed\xf6\x9d\xf2\xfb\x0f\x95\xeb\x1b\x88B\x18\'o\xf9my\xf7C\xf9m\xd1\xdd\x9d\xaf\xcd\x94*\x0b\xbb~\x16\x89\xd0V^/zwW\x02\x0b\xf3&y<\x13\xf3V~\r\x83/\x94\xf9c\xe7\xf6d`\xf2T\xcc\x0f\xc7Q\x9eQ\x91`\xa4\xb7\xa5\xe48\xeb\xeb;{,t\xb6H5\xa6\xda\x16\xd3\x0c%\xcdM\xe0\x95z\x01:\xdb\x066\xcc+\xa2N\xfa\x06\xceZ\x89\xb6\xf2\x87\x9f+K\xb7\x905\x90*\xf7\x88\xc6\x852mk\xdd\xfdm\x06\xc9\xe6.\xce{\x1bk\xee\xdd\xa2{\x87\x12\xd2\x9d\xbd\\\x9b.\xc1\xc5\xee\xf4cw\xe75m\x17\x18\x12j\xd1\xc3&\xdb\x18\xfe\xc5:c\x87\xd9d,\xa5!\x88I{"\xcfq\x19;\xabs\xfb\x84\xd0\x8e\xc4\x9et4[\xc5\x1e1\xc4R\xd5\xf3\x8em\x89W\x1c,J\xea\xb0\x8f\xde\xf8a\xaf\xc0\xad\xbd\xb6c\x0e\x1b{\xffq \xf9mbd\xc4\xc9\xd0+i>\xa6\xa6\xc4\xa2\x94\x93V\x0ew\xc4\xa6\xa6\xf6H\xd1]-D\x7fs\xa2\x7f@\x8a\xfe\x84\xd8\x94\xa6\xe6C\xb1\xdf\x8fs=\xd9\x9d\xdc7\x9c\xcc\xe4\x0f&s\xe4\xc8\x84\xa5d\xb8\r\xf04L\x8b^\r6\xd6\x9c\x9c\xaa+]-4\xea\xack\xd4\xddB\xa3\x7f\x1e\xe9\xfdS\x85\xc6\x14\x1e\xea\xa3\xf0f\xf1\x9f\x96\xb5\xef\x13\xd6\x0f\xf1\x0bv\xaf\x9e2\xd2\xbcI\x90\x8d\x07\xf81p\xe6\xf4\xc9\x81!<"O\xe0\xfa\\\x8c\x8c\xea8\x1fn\xbc\xbf\xd5\xc6\x86\x9e\x06\xaa\x19:P\xe0\xdf\xdc4N;v\xb3_#+\xc4\xae\xfb\x1aw=\xd0b\xd7\xde\\\xde\x9e\x18<\xda}\x02I\xab\xdb\xa2\x955\xed:\xae\xa6\xed,~\x89\xde\x82GY\xae\x8ed\xc9\n\xd9XpgX\xb1S\xd9\xa4\xa5\xfe@;v\x86\xe2\xbej!\xee$|\xa2\r*\xb9\xbc\x06\x1d\x8f8\xa6r\x0c\x99\xdc$\xd0\xca\xaa\x19\xda\xbf\x1b\xb7s\xb4^\xd8\xd2\xd9h\xcb\xc1\x16\x9b\x7f+6\xe6f\xf3~(r\xfc\x08j\x1c\x0f\xd1\x9b\xf2\xf4lp\xa8w`\x90\xc2\x90!\x7f}s\xac\x8f\x9e\xc9=\xc2|@\xb7K\xe6(m\xe8%\xda\xd9Jey\xda!1xj\xa9\xb4Zf\x89n\xa8\x96\xf0@\x83\xde_\t\xbdq+oX\x08\xce\x18\x8fD\x06wu>\xa2\x84w\xf7\x07w5\x11\x8d\xa4(H\xf1\xe4@\xa3\xf5\x87Z\'\xf9Q\xde"\xe7\xa4A\x96\xd8\xe6` \x00\x19/nt7\xee\xdb\xd9\xd1b\xe3Ae\x8c\xb7J\x8c\x0c\x98\x11y)\x997\x81\x96\x17\x04\x86\xa8\xe1:\xfa!\x85\x1e\neL\xf9-A\xf0 \xd9\xc9\x04U\x02\x17"\xf0\xa4\x1b\xfc\x02\x1cM}\xee\x1b#\x97\x998\xdb\x0fD5\xc7\xa8\x13\x00&%hK\x04\xad\x15.\xba\xbb\x05\xf7\xcdKwi\xd5-.\x11\x91\xf0\xdf\x10\xa8ZB;\x00\xbf\xa8\xee\xcc\x81\x16J`\xf5VJn\xf1V\x80\xa2\x08k\x1e\xd4\x8c\x03E}\xbab%\x10!\xbb\xfd\xefG\x06{\xd9\x17,\xf6%Z\x11j\x03\xba\x7fg\x19z\xcfd\xcc\xbf>\x1c\xb6\x01Y\xbf"N\x94_\x96m\xb6\x13\x9bL\xd0\x7f\xfb\xda\xe3\xf1\xa9=\xcc\x06\x07\x04\'\xe9\xe9\xec\xea\x88\x07\xb6\x9b\x19\x05\x8d{L\xd1\xd4\xb4B\xc5\xca\xb8i\x02i\xd0\xd4\x91a\n\xd8T\x86Y\xe4Y\x9bH\x95\xa2J\xcaR->\x8av\x8e\xda\xe3\x1b\xd5gs\xee\xc3\x17\xde\xd2\r\xd8\x8b\xae\xef\x96~\x97\xbd\xb26\xbfY}\xf20dP\x19\xe2?,\xb06a\x8c\x1en\x93\xa1\xc6C\xfa\x1b\xb6\xe7\xd0!\xc3Fz\x82\xa2\x91\xe5Z\x1e$V\x19\xd1\r\x8bXT\x1a\x9c\xcb\x0cH\x94\xaftB\xec!\x05\x97w\xb6\xc9\xc5\xb76\xdc;wA\x9d\x88q\x94\xae\xbbW\x1e\x13\xbdz_\xaal\x16j\xef\xeeJ\x8a@\xfdmi\xa1\xb66OZF\x951\x15T\x10;\xe3\xe8\xe4\xb7^\x92\xd0\x1ejN\xd0\x19\xf8\x10\t)[\xb3$\xd4h\xbd\x94A*J\xd0V3*eKu\xe3fm\xfeR\xb4\xd7K\xf7\x11{x\xb3Q\xbd\xb5\x1a8\x08/"|"\x0b|A\x14\xee\xf6\xf89?\xde\x14\xdb\xf3R\xec\x10\r$2I5%oA\xaa?\x86P\x10I\xe3Vb\xbd\xb9B\xf5\xca\x0b\xd0\xde\xda\xf2\xb5@\xa6e+\xe0Z=\xf2\x9d\x9c\x81\x00\x01\xb2S\xed\xf1\x08a\xc2\x12\xdb\xb1@\xaa\x91\tA\x16\xc8\xa9\x08\xe2\x07N\xf51d\x15\n1\xd1VA\x1a\xecN\x8b[\xd5\x9d-wf\x1b\xa2h\x80XY\xf5\xd3A\x0eC\x1bW\xbc\x8b\x9b\x81|\xdaG\x81\xdb`\x1e\xb2\x8ex\xd8\x08o\xef<\xd8\xd1\x11oJ\x8e3\x94\x8f$1\xe5\x98\xe0\x9bv\x9d\x01eQ\xa5\x869!\xe3_}^p\x17\x96+w\x9e\x83hF\xb3\x94\x08\xf3\xea\x9b\xea\xda|\xe5U\x01n\xf9(\xda@+3\x1d-\xc0\x11\x1e\xa9?_\xc4\x971\\\xc9(\xd5\xeb\xa9\xbb#\xee\xc7I\xbc#\x1f\xc7\x1b\x95\xef\xd7\xad<O\xd9,e\x10\xfeQ\x92\xc0\xec\x8c\x02\x87\x8at\xf1s\xd7\x9f\xc0\x80#\xcf\xe7)\\\xb3\xcb\xb2\xaa\xe4\xbc\x08S\x9a\xb5FEI\xc5}_E\x9fI\x9f\x89\x99\x85_\xe0)\'Z\xdebj\xf4!*\x11\xae\x96A\n\xc1N\x12]\xe9\xb4z!\xb7\x12\x13\xaa!\x1c\x10\x93\xf9\x02\xb0\x9a\x9c\x8aG\xef$\x81J\xb18\xcd\x1d1\xa1G,\xa2u\xf3\x96\x02\x0el#/G9Q_\x02{\x89\x96\xc3$\x0b\xd3\xa44&\xd1\xf0\x8a\xc4U\x14\xbd{\xef\x85\xd4\x19)(\x11\xc9\xbd\xb8\xe2=\xbb\x1f5\xa1\x95\xcc\x16UOX*\xcd;\x17\x98v>\x1e\xff\x94\xc3\x15\xff\x18\x80\xe5\xb8\xad\x00T\x15iA\xa4:A\xe6U+\xcb\xad\xba\xe6\xf5\x11\xe1\xe7UL\x0f\xc8_w\xf7\x86\x1c\xc4\xab\xbf\xcf\xa0^>\xe9x\xd1\xd8D\xd6\xd6\xdd/\xab\xb1\xc1\xffh\xa5\x91+\xbf\x1d\xeea\xe7\xce\x7f\xc2\x0c\xe1\xf9\xbc\xa1i\xe2x\xc0\xc2\x88\x04\xbd\xf1\xb7\xde%1Y+c\xc8_eX\xe3uC\xfc\xc3\x03$\xf0\xc2\xa6\xb4\x82\xdc/\xec\x92\xd8\xf0\'\x19$\xb5\xfa\x8b\x94\x00\xef\x04\xfa8\x9czB\xda\x18\xd7i\xe6\t\x07Qy\xfa\xd2\x90\x0eW\x1fU\xffs\xa3\xbcUD\xef\xf5\x91Wh\xf8WY0lre\xb4\xb1\x82\x07hd\xa3\xf9^5\xd2j*\xc0DU\xe6f\x8ak\x9a\x8f\x82\xe1\xe8\x0f\xd6B#\x02\x1d\xaf\x10[\xa3WS\x80\xadP\xc7(0\xc2c\xd2?\x12\x9b%k\xf0J\x0b\xb5\xc2=\xef\xea\x9a\xb7\xf1\xa0rk\x17S\x98\xb7y\xbd\x05\x02\x04\xe8\xf9\x19\xdb/\xceT>\x81\x05G\xfd\xa3\xa4\xa0_X\x9c(z\x00\xeb\x91\nCh\x83l\xa5n_\x8f\xed\x86\xb4\x13\xee\xc4\xac+\x15\x978[-\xdc\xa8\xbe|W\xf9\xe5\'yL\xd3R|0\xf2\xca0\x94\xb7\x9e\xfd\xb7\xb0\xce&\x9b\xbb\x0e\xdb\xcbDC:\x9c\xe8\xc8L1\xec\x88,\xcdh\x8e\x95\xed\x192\x1d\xde\x04\xaa\x03\x8aC\x14A\x9eI\x88\xd3\x15\xb4\xe5\xa0\x1b\x04@.\r\x90\xde\xa5\xdc\x9c}Gd\xe0r\tM\x11X[}|\xdf\xef\xc5\xc5\x827]l\xf6\xaeP\xcf\xd28\xcf\xb7wA8\x9d\x954S\x16\xaa\x14\x04!\xe2?\x9dK\x9c\xc2\xec\xaa\x0b\x9c\x8a$&\xa8Sm\xe9\x8d\x8c\xb2\xf7\xf2\'o\x05iy\x1f.\x91iI\x9aAE\x89^\x85],\x8b*\xd4\xd4\r\x05\xb2+,\xa5q\xc5\x0cz\x11\xe0\x0b\x8aPc\x86X\xcd0\xf2\x94\x7f\xaa\x9e\xa6\x03\x04\xd5\xe6\xdaD\x94#I\t\xb2\x1bCr\xed\xe2\x07w\xbaT\xb9\xb4V\xbb\xb5\xd8X\xac\x8d\x8chH\x8a\x92\xd8\x18\xc4S\xee\x85]\xaaO\x8b\xb2\x8b\xb1\xc9\x13\xa7\xfb\x92\x03G\x86\x8e\xd7\xcf1\xfc\x82\x95%\x13vB\xa2\x0b\x8d\r\xc9\xdd\xbd[\xefB\x02F\xfc\xe2\x15m\x91\xe85\xde\xa9\x9f\xcb\xc1\xbb\xc9V\x14\xba\xa1\x83\x8f\xa9\x9cf;:\x11\xc9Y=\x12l\xce\xd1\xf8\xf11->\xc3\xbf\xa3^\xed\xb7\xe7\x00f,\xc2\x1cL\x17c\xc4=\x80\xb6\xa6\x93\'\xedi\xf0\x00\x94\xd7n\xbeDw\x91H#\x9b\xb6[zW~Wr/\xbdt\x9f\xdd\xa43\x15\xe8\xbe\xf4\xaa\x0e?\x1f\xab\x9e\x10\xfeM"\x9b\x93\x12]B\x12\xa6 x\x18h}j\xec\x9fg\xb2`\xe4\x11\x07\xab\\\xc7{\xa9F:\xe6mMW\x16/\xfbL\xec\xc9uw\xb1T\xdeZ\xaf\xbc~A\x07\x8cB-:\x1e\x8bx2\xdc\xb0\x07\xa5eg\xdbO\x9f\x1d\x1a8;\x94<u\xe4do<A\x0f\xa4:\xbd\xba\xe5\xa0\xe0|\xb8\x0b^B\xa6k\xe3\xca\x84\xc5\xb2\x01!&UP\x90\xf2\xdc\x19\xe4\xf0\xfev\xf9\xc3\xcf\r\xda\x8834\xb1\xce\x9d\xdb\x94\xf7\x9af\x85`\xf7\x04x\x1e7\xc1\xb08 \x8b\x10\xb6=\x96\xc8\xeb#\xb1f\x92x$\x9f\xc7\nY\x8d6\xfc\xa2\x98\xe9f=D\x19\xce<\xc1\x9cV}\xb0R\xde\xba\xf2\xb1\xfc\xe8\x86\xa1O\xbe\x00k\x11"#\xb4\x1f\x91\xc30.\xba\xc0\xb7\xca\xc8\x08foK\xb0R\xaa\xba\xb4\x8a\x9e,i)M\x8b\xe2\xf8Oz\x9d\x80T.g!\xb1\x03\x8e\xfa\xa4\x946\xf7\x1dO\xc5\x83X\x9c9}z\x88}\x19j\xe2\xcb\x17\xb4\xddr2\x18v\x99\xee\xe4\x86\xd1\x98|V\x9ev\xf2\x9a\x9a\xa2s\xbf06\xb2\xa4::\xbaXm\xb9\xe8\xfeR\x94c\x87\xbb8\x0f\x8b\x9bR\x12`q\x01B\xbb\xa4\x98\x01\x98H\xa8\xc6\x002\x18/\xcd\x08\x8f\xb1\xd8\xf0\x04Ke\r\xc3\x92\xdfE\x98\xa3;V$-Q\x13\x17\xb7\xbck3\xde\xca\x93\xf0\xb0T\xfa8\x04\x1bLf\xd5kEwz\x16=\x04Np\xaf]\x0c\xcf\x15ES\x8a\xf8!!fq\x94Ds\xc0\xd3i\x80 \x1c\x00\xfbS\x81?\xc8\x0b\\Ie\x99\x92\x96gP\x8a\x16\x05a\x04\x00\x13y\xf5\xe1\x8cT\x87\x18\xd7\x95{\x18\xcd\xeb\xc3XTB\xeb`\xc4&E\x89\x04\xc6\xc6\x13(\xc9\xdcTrR\xf8\xefpGwz\xcaO\x96\xe8V\'\x8d1\x1e\x9c-\x88^%\x03\'\xfb\x07\xda\xaeH4bV\x1a\xf8\x07\x06_\xf2g\xd87\xa0ie\xe3\x05 E\xeaN\x93\xf6\xdb\x9d\xca\xf6\xbc{\xf3F\xf5\xf1:\x90\xc7\xffn\xd0\xd2\x08\x19V\xe4pg\xd3\xd4\xaa\xf3qQ\x00\xd0\xca\xcfI\xea\x9e\xf23Vp\xc8\xe1\x03.\x06\x82\xa7\xf0R\xb0\xce\x87\x90\xc7\xdb\x91a2\x1a0\xfa\xec\xc1\x93\xc3\x136\xb7\xda\x9b\x80\x0e\xc4\x83\x8e\x94\xa23\xa6\xdf\x0e\x82\xe9U`\x9a<\xf0\x1ek\xcc)I;\x84^\xa0\x1d\xd4\xc0fg\xa4*\x8d\x00\x12\xf0\r\xd93\xe8\\=\xaa\x1cm6\xf5\xc7\xfb\x95?! \x9d>\x01\x89\x87\x95@-D(\xe53\xf8\x80\xe9\xfb$0\xf8\xb6\x03\xa8\xbb\xf3\n\xcc\x93\xea<\xfc^\x03\xc2S\xbb\xb7X]\x7f\xdfLCio\xffc^\xbb8\xd8J\x18\x80\xb0\xf6\x88\xb2\xf1P\x03cLM\xd3W\x19\xcdH!\xa3A\xceG\x83l\x92nB\x90\x80\xfa\x0b\x8b\xe5\xdd;"`\xf0\x88\xb76+\x9bR\xad\xf0\x00 \xd4,1\xf8\xa0(\x86\x9c\x06\x99\xf1\xff\x03\x14\xc4\x90\xb2').decode('utf-8')
REMOTE_ALLOWED_TASKS = ['text']
REMOTE_MAX_IMAGES = 0

REMOTE_ROOT.mkdir(parents=True, exist_ok=True)

'''
Share existing state when the API cell is executed again.
重跑 API Cell 時保留原有工作及鎖定狀態。
'''
REMOTE_JOBS = globals().setdefault('REMOTE_JOBS', {})
REMOTE_JOB_LOCK = globals().setdefault('REMOTE_JOB_LOCK', threading.Lock())
REMOTE_ENGINE_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
REMOTE_QUEUE = globals().setdefault('REMOTE_QUEUE', queue.Queue(maxsize=REMOTE_QUEUE_CAPACITY))
assert REMOTE_QUEUE.maxsize == REMOTE_QUEUE_CAPACITY, 'FIFO 容量已變更，請先重新啟動 Kernel 及遠端服務。'
REMOTE_UPLOAD_SLOTS = globals().setdefault('REMOTE_UPLOAD_SLOTS', threading.BoundedSemaphore(REMOTE_UPLOAD_CONCURRENCY))
REMOTE_RESULT_TTL = REMOTE_RESULT_TTL_HOURS * 3600
REMOTE_RESULT_BYTES = REMOTE_RESULT_LIMIT_GIB * 1024**3
REMOTE_MIN_FREE_BYTES = REMOTE_MIN_FREE_GIB * 1024**3
REMOTE_STALL_SECONDS = REMOTE_STALL_HOURS * 3600

'''
Verify the CUDA backend and existing model without loading another model.
確認 CUDA 與已啟動的模型後端，不另外載入第二份模型。
'''
def remote_gpu_status():
    try:
        import torch
        if not torch.cuda.is_available():
            return False, 'Kaggle CUDA GPU 不可用；請啟用 GPU 並重新執行模型初始化 Cell。', []
        qwen = REMOTE_KIND == 'image_multi'
        yue = REMOTE_SERVICE_ID.startswith('06_YuE2_')
        if yue:
            selected_gpus = [globals().get('GPU_ID', 0)]
        elif qwen and globals().get('QWEN21_STATE', {}).get('mode') != 'bf16_dual':
            selected_gpus = [0]
        else:
            selected_gpus = [0, 1]
        available = torch.cuda.device_count()
        if any(not isinstance(index, int) or index < 0 or index >= available for index in selected_gpus):
            return False, f'需要 cuda:{selected_gpus}，但實際只有 {available} 張 CUDA GPU。', []
        devices = [f'cuda:{index} {torch.cuda.get_device_name(index)}' for index in selected_gpus]
        if qwen:
            state = globals().get('QWEN21_STATE', {})
            if state.get('pipeline') is None or state.get('manager') is None:
                return False, 'Qwen Pipeline／GPU 記憶體管理器尚未初始化，請先執行模型初始化 Cell。', devices
            if state.get('device') != 'cuda:0':
                return False, 'Qwen Transformer 沒有設定為 cuda:0。', devices
        elif REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
            pipeline = globals().get('pipe')
            if pipeline is None or pipeline.lazy_load or pipeline._mula is None or pipeline._codec is None:
                return False, 'HeartMuLa 雙 GPU Pipeline 尚未完整載入，請先執行模型載入 Cell。', devices
            for name, module, index in (('HeartMuLa', pipeline.mula, 0), ('HeartCodec', pipeline.codec, 1)):
                parameter = next(module.parameters(), None)
                if parameter is None or parameter.device != torch.device(f'cuda:{index}'):
                    return False, f'{name} 未位於指定的 cuda:{index}，請重新載入模型。', devices
        elif yue:
            gpu_id = globals().get('GPU_ID', 0)
            process = globals().get('SERVER_PROCS', {}).get(gpu_id)
            if process is None or process.poll() is not None:
                return False, f'YuE2 CUDA{gpu_id} 常駐服務尚未啟動。', devices
            if f'CUDA{gpu_id}' not in [str(value) for value in process.args]:
                return False, 'YuE2 常駐服務沒有使用預期的 CUDA 啟動參數。', devices
        else:
            base = globals().get('BASE')
            if not base and REMOTE_IS_LTX:
                base = f"http://127.0.0.1:{globals().get('COMFY_PORT', 8188)}"
            if not isinstance(base, str) or not base:
                return False, 'ComfyUI 尚未啟動，請先執行模型服務 Cell。', devices
            with urllib.request.urlopen(base.rstrip('/') + '/system_stats', timeout=4) as response:
                stats = json.load(response)
            comfy_devices = stats.get('devices', [])
            if not any(str(item.get('type', '')).lower().startswith('cuda') or
                       str(item.get('name', '')).lower().startswith('cuda:')
                       for item in comfy_devices if isinstance(item, dict)):
                return False, 'ComfyUI 沒有回報 CUDA 裝置，已拒絕 CPU-only 推論。', devices
        return True, 'CUDA 後端已就緒（不代表所有權重常駐 GPU 記憶體）。', devices
    except Exception as error:
        return False, f'GPU 檢查失敗：{type(error).__name__}: {error}', []

'''
Verify every global required by the selected Notebook generation engine.
驗證所選模型的生成程式所需變數與設定已完成初始化。
'''
def remote_inference_ready(task):
    namespace = globals()
    common = {
        '06_HeartMuLa_': ('ROOT', 'pipe'),
        '06_YuE2_': ('ROOT', 'GPU_ID', 'PORTS', 'SERVER_PROCS'),
        '06_Z-Image-Turbo_': ('ROOT', 'BASE', 'LOG_PATH'),
        '06_Qwen_Image_': ('MODEL_MODE', 'GPU0_MODEL_GIB', 'GPU_ID', 'QWEN21_STATE'),
        '06_LTX-2.5_': ('ROOT', 'ENGINE_PATH', 'COMFY_PORT', 'COMFY_LOG', 'COMFY_PROC',
                          'ltx_available_ram_gib', 'LTX_MIN_AVAILABLE_RAM_GIB', 'importlib'),
        '06_MiniMax_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                              'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                              'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                              'QUANT_SIZES_GIB', 'ready', 'choices', 'check_choice'),
        '06_HyperFlow_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                                'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                                'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                                'QUANT_SIZES_GB', 'ready', 'choices', 'ENCODERS',
                                'ENCODER_FILE', 'MODEL_FAMILY', 'WEIGHTS_IN',
                                'TEXT_LOADER_NODE', 'PROJECTION_FILE'),
    }
    required = next((names for prefix, names in common.items()
                     if REMOTE_SERVICE_ID.startswith(prefix)), ())
    missing = [name for name in required if name not in namespace or namespace[name] is None]
    if missing:
        return False, 'Notebook 尚未完成初始化，請先執行對應的模型／服務 Cells；缺少：' + ', '.join(missing)
    if task not in REMOTE_ALLOWED_TASKS:
        return False, '此模型不支援生成模式：' + task
    if REMOTE_SERVICE_ID.startswith(('06_MiniMax_H3_', '06_HyperFlow_H3_')):
        config = namespace['TASK_CONFIGS'].get(task)
        if not isinstance(config, dict):
            return False, f'{task.upper()} 尚未設定；請先執行 Notebook 對應的模式設定 Cell。'
        if namespace['H3_ACTIVE_CONFIG'] != namespace['H3_DESIRED_CONFIG']:
            return False, 'GPU／Offload 設定與已啟動 ComfyUI 不一致，請重新啟動模型服務 Cell。'
        quant = config.get('quant')
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            variant = 'ref2va_pruned' if task == 'r2v' else 'fl2va_pruned'
            expected = (variant, quant)
            expected_file = f'minimax_h3_{variant}-{quant}.gguf'
        else:
            variant = 'ref2va' if task == 'r2v' else 'fl2va'
            encoder = config.get('encoder_variant')
            expected = (variant, quant, encoder)
            expected_file = f'minimax_h3_{variant}_hyperflow_flash_14b-{quant}.gguf'
            if encoder not in namespace['ENCODERS']:
                return False, f'{task.upper()} 的 Encoder 設定無效，請重新執行模式設定 Cell。'
            if Path(namespace['ENCODER_FILE']).name != namespace['ENCODERS'][encoder][1]:
                return False, 'GPU 1 Encoder 與此模式不一致，請重新執行模型下載／啟動 Cells。'
        if namespace['H3_MODEL_CHOICE'] != expected or namespace['H3_ACTIVE_MODEL'] != expected:
            return False, f'{task.upper()} 尚未載入相應模型權重；請設定模式並重新執行模型下載／啟動 Cells。'
        if namespace['DIT_NAME'] != expected_file:
            return False, 'ComfyUI 使用的模型檔名與任務不一致，請重新載入正確模型。'
    elif REMOTE_SERVICE_ID.startswith('06_Qwen_Image_'):
        state = namespace['QWEN21_STATE']
        if state.get('mode') != namespace['MODEL_MODE']:
            return False, 'Qwen MODEL_MODE 與已初始化的模型模式不同，請重新執行模型初始化 Cell。'
        if state.get('gpu0_model_gib') != namespace['GPU0_MODEL_GIB']:
            return False, 'Qwen GPU0_MODEL_GIB 與目前載入設定不同，請重新初始化 GPU 記憶體管理器。'
    elif REMOTE_IS_LTX:
        if not Path(namespace['ENGINE_PATH']).is_file():
            return False, 'LTX 模型引擎檔案不存在；請先執行模型與 Workflow Cell。'
        if namespace['COMFY_PROC'].poll() is not None:
            return False, 'LTX ComfyUI 行程已停止，請重新執行啟動 ComfyUI Cell。'
    return True, '模型初始化與所選模式已就緒。'

'''
Validate client parameters before they occupy a FIFO slot.
在進入 FIFO 前檢查 client 參數，避免排隊後才因設定錯誤而失敗。
'''
def remote_validate_fields(task, fields):
    required = {'task'}
    if REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
        required.update(('lyrics', 'tags', 'duration', 'seed'))
    elif REMOTE_SERVICE_ID.startswith('06_YuE2_'):
        required.update(('style', 'lyrics', 'duration', 'steps', 'lm_seed', 'seed', 'cot'))
    elif REMOTE_KIND == 'image_text':
        required.update(('prompt', 'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_KIND == 'image_multi':
        required.update(('prompt', 'negative_prompt', 'sampler', 'reference_size',
                         'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_IS_LTX:
        required.update(('prompt', 'width', 'height', 'seconds', 'seed', 'image_strength'))
    else:
        required.update(('prompt', 'width', 'height', 'seconds', 'steps', 'seed', 'ref_image_size'))
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            required.add('audio_shift')
    missing = required - fields.keys()
    if missing:
        raise ValueError('請求缺少必要參數：' + ', '.join(sorted(missing)))
    integer_fields = ('seed', 'lm_seed', 'width', 'height', 'steps', 'duration', 'reference_size')
    float_fields = ('seconds', 'cfg', 'image_strength', 'audio_shift')
    numbers = {}
    for key in integer_fields:
        if key in fields:
            raw = fields[key]
            if isinstance(raw, bool) or str(raw).strip() != str(int(raw)):
                raise ValueError(f'{key} 必須是整數')
            numbers[key] = int(raw)
    for key in float_fields:
        if key in fields:
            numbers[key] = float(fields[key])
            if not math.isfinite(numbers[key]):
                raise ValueError(f'{key} 必須是有限數值')
    for key in ('seed', 'lm_seed'):
        if key in numbers and not -1 <= numbers[key] < 2**32:
            raise ValueError(f'{key} 必須介於 -1 至 4294967295')
    for key in ('width', 'height'):
        if key in numbers:
            size = numbers[key]
            divisor = 64 if REMOTE_KIND == 'image_text' else 32
            if size < 1 or size > 4096 or size % divisor:
                raise ValueError(f'{key} 必須介於 1～4096 且為 {divisor} 的倍數')
    for key in ('seconds', 'duration'):
        if key in numbers and not 0 < numbers[key] <= 3600:
            raise ValueError(f'{key} 必須大於 0 且不超過 3600')
    if 'steps' in numbers and not 1 <= numbers['steps'] <= 200:
        raise ValueError('steps 必須介於 1～200')
    if REMOTE_SERVICE_ID.startswith('06_HyperFlow_H3_') and numbers.get('steps') not in (4, 6, 8):
        raise ValueError('HyperFlow steps 只支援 4、6、8')
    if 'image_strength' in numbers and not 0 <= numbers['image_strength'] <= 1:
        raise ValueError('image_strength 必須介於 0～1')
    if 'reference_size' in numbers and not 64 <= numbers['reference_size'] <= 2048:
        raise ValueError('reference_size 必須介於 64～2048')
    if 'audio_shift' in numbers and not 0.01 <= numbers['audio_shift'] <= 100:
        raise ValueError('audio_shift 必須介於 0.01～100')
    if fields.get('ref_image_size', 'match') not in ('match', 'max'):
        raise ValueError('ref_image_size 只支援 match 或 max')
    if REMOTE_KIND in ('video', 'image_text', 'image_multi') and not str(fields.get('prompt', '')).strip():
        raise ValueError('PROMPT 不可為空白')
    if task == 'r2v':
        matches = [int(number) for number in re.findall(r'<Picture\s+(\d+)>', str(fields['prompt']), re.I)]
        if any(number < 1 or number > int(fields.get('_image_count', 0)) for number in matches):
            raise ValueError('R2V Prompt 的 <Picture N> 編號超出已上傳的圖片數量')

'''
Report whether the model is ready for CUDA inference.
回報模型是否已具備 CUDA 推論條件。
'''
def remote_backend_ready():
    return remote_gpu_status()[0]

'''
A Python thread cannot safely terminate an already running CUDA inference.
Python thread 無法安全強制終止正在執行的 CUDA 推論。
'''
def remote_stalled():
    with REMOTE_JOB_LOCK:
        return any(record.get('status') == 'running' and
                   time.monotonic() - record.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
                   for record in REMOTE_JOBS.values())

'''
Remove obsolete input files once the model no longer needs them.
模型不再需要輸入檔案時立即移除，避免參考圖片長期佔用空間。
'''
def remote_remove_inputs(job_id):
    folder = REMOTE_ROOT / job_id
    for file in folder.glob('*'):
        if file.name.startswith('input.') or (file.stem.isdecimal() and file.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}):
            file.unlink(missing_ok=True)

'''
Keep only finished results; never delete active jobs or in-progress downloads.
僅清理已結束工作，絕不刪除排隊、推論或正在下載的檔案。
'''
def remote_cleanup():
    now = time.time()
    with REMOTE_JOB_LOCK:
        finished = [(job_id, record) for job_id, record in REMOTE_JOBS.items()
                    if record.get('status') in ('done', 'error', 'expired') and
                    not record.get('_downloads')]
        finished.sort(key=lambda item: item[1].get('finished_at', now))
        sizes = {}
        for job_id, record in finished:
            folder = REMOTE_ROOT / job_id
            sizes[job_id] = sum(path.stat().st_size for path in folder.glob('**/*') if path.is_file()) if folder.exists() else 0
        total = sum(sizes.values())
        for job_id, record in finished:
            old = now - record.get('finished_at', now) > REMOTE_RESULT_TTL
            if old or total > REMOTE_RESULT_BYTES:
                shutil.rmtree(REMOTE_ROOT / job_id, ignore_errors=True)
                total -= sizes[job_id]
                record.update(status='expired', error='結果已過期或因儲存空間不足而清除，請重新提交。')
                record.pop('_path', None)
                record.pop('_last_frame', None)
            if record['status'] == 'expired' and now - record.get('finished_at', now) > 2 * REMOTE_RESULT_TTL:
                REMOTE_JOBS.pop(job_id, None)

'''
Sweep completed records periodically without interfering with inference.
定期整理已完成資料，不干擾單一 GPU worker 的推論。
'''
def remote_maintenance():
    while True:
        try:
            remote_cleanup()
        except Exception as error:
            print('API cleanup warning:', type(error).__name__, str(error)[:200])
        time.sleep(300)

'''
Serve only the generation API without exposing notebook controls.
只提供生成 API，不公開 Notebook 的管理功能。
'''
class RemoteHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        return

    def send_json(self, code, payload, retry_after=None):
        data = json.dumps(payload, ensure_ascii=False).encode('utf-8')
        self.send_response(code)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(data)))
        if retry_after is not None:
            self.send_header('Retry-After', str(retry_after))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        path = urlsplit(self.path).path
        if path == '/health':
            with REMOTE_JOB_LOCK:
                running = sum(record.get('status') == 'running' for record in REMOTE_JOBS.values())
            gpu_ready, gpu_message, gpu_devices = remote_gpu_status()
            requested_task = parse_qs(urlsplit(self.path).query).get('task', [REMOTE_ALLOWED_TASKS[0]])[-1]
            inference_ready, inference_message = remote_inference_ready(requested_task)
            if gpu_ready and not inference_ready:
                gpu_message = inference_message
            self.send_json(200, {'ok': True, 'api': 'kaggle-generation-bridge-v3',
                                 'service_id': REMOTE_SERVICE_ID, 'task': requested_task,
                                 'backend_ready': gpu_ready and inference_ready, 'gpu_ready': gpu_ready,
                                 'gpu_message': gpu_message, 'gpu_devices': gpu_devices,
                                 'stalled': remote_stalled(), 'running': running,
                                 'waiting': REMOTE_QUEUE.qsize(), 'queue_capacity': REMOTE_QUEUE.maxsize})
            return
        if path == '/schema':
            self.send_json(200, {'kind': REMOTE_KIND, 'tasks': REMOTE_ALLOWED_TASKS,
                                 'fields': REMOTE_FIELDS, 'image_limit': REMOTE_MAX_IMAGES,
                                 'upload_field': 'images' if REMOTE_MAX_IMAGES else None,
                                 'queue_capacity': REMOTE_QUEUE.maxsize,
                                 'model_switching': 'configure in notebook first'})
            return
        pieces = path.strip('/').split('/')
        if pieces[0] != 'jobs' or len(pieces) not in (2, 3):
            self.send_json(404, {'error': 'Not found'})
            return
        job_id = pieces[1]
        with REMOTE_JOB_LOCK:
            record = REMOTE_JOBS.get(job_id)
            item = dict(record) if record else None
            if item and item['status'] == 'running':
                item['stalled'] = time.monotonic() - item.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
            if item and item['status'] == 'queued':
                waiting = [key for key, value in REMOTE_JOBS.items() if value.get('status') == 'queued']
                if job_id in waiting:
                    item['queue_position'] = waiting.index(job_id) + 1
            if item and len(pieces) == 3 and pieces[2] in ('file', 'last-frame') and item['status'] == 'done':
                record['_downloads'] = record.get('_downloads', 0) + 1
                downloading = True
            else:
                downloading = False
        if item is None:
            self.send_json(404, {'error': 'Unknown job'})
            return
        if len(pieces) == 2:
            for field in tuple(item):
                if field.startswith('_'):
                    item.pop(field)
            self.send_json(200, item)
            return
        if not downloading:
            self.send_json(404, {'error': 'Result not available for this job'})
            return
        try:
            file_path = item.get('_path') if pieces[2] == 'file' else item.get('_last_frame')
            if not file_path or not Path(file_path).is_file():
                self.send_json(404, {'error': 'Output is no longer available'})
                return
            result = Path(file_path)
            self.send_response(200)
            self.send_header('Content-Type', 'application/octet-stream')
            self.send_header('Content-Disposition', f'attachment; filename="{result.name}"')
            self.send_header('Content-Length', str(result.stat().st_size))
            self.end_headers()
            with result.open('rb') as stream:
                shutil.copyfileobj(stream, self.wfile, length=1024 * 1024)
        except (BrokenPipeError, ConnectionResetError):
            pass
        finally:
            with REMOTE_JOB_LOCK:
                record['_downloads'] -= 1

    def do_POST(self):
        if urlsplit(self.path).path != '/generate':
            self.send_json(404, {'error': 'Not found'})
            return
        supplied_id = self.headers.get('X-Request-ID', '').strip()
        try:
            job_id = uuid.UUID(supplied_id).hex if supplied_id else uuid.uuid4().hex
        except ValueError:
            self.send_json(400, {'error': 'X-Request-ID must be a UUID'})
            return
        with REMOTE_JOB_LOCK:
            existing = REMOTE_JOBS.get(job_id)
        if existing:
            self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                 'download_url': f'/jobs/{job_id}/file', 'reused': True})
            return
        if remote_stalled():
            self.send_json(503, {'error': 'GPU worker appears stalled; inspect the Notebook before accepting more jobs'}, 60)
            return
        gpu_ready, gpu_message, _ = remote_gpu_status()
        if not gpu_ready:
            self.send_json(409, {'error': gpu_message})
            return
        if shutil.disk_usage(REMOTE_ROOT).free < REMOTE_MIN_FREE_BYTES:
            self.send_json(507, {'error': 'Kaggle disk space is low; clean old outputs before retrying'})
            return
        if not REMOTE_UPLOAD_SLOTS.acquire(timeout=1):
            self.send_json(503, {'error': 'Upload slots busy; retry later'}, 5)
            return
        staging = None
        try:
            self.connection.settimeout(120)
            length = int(self.headers.get('Content-Length', '0'))
            if not 0 < length <= 120 * 1024 * 1024:
                raise ValueError('Request must be between 1 byte and 120 MiB')
            body = self.rfile.read(length)
            if len(body) != length:
                raise ValueError('Incomplete request body')
            content_type = self.headers.get('Content-Type', '')
            fields, files = {}, []
            if content_type.startswith('application/json'):
                fields = json.loads(body)
            elif content_type.startswith('application/x-www-form-urlencoded'):
                fields = {key: values[-1] for key, values in parse_qs(body.decode('utf-8')).items()}
            elif content_type.startswith('multipart/form-data'):
                message = BytesParser(policy=policy.default).parsebytes(
                    b'Content-Type: ' + content_type.encode() + b'\r\nMIME-Version: 1.0\r\n\r\n' + body)
                for part in message.iter_parts():
                    name = part.get_param('name', header='content-disposition')
                    data = part.get_payload(decode=True)
                    if name == 'images':
                        files.append((part.get_filename() or 'upload.png', data))
                    elif name:
                        fields[name] = data.decode('utf-8')
            else:
                raise ValueError('Use application/json, form data, or multipart/form-data')
            if not isinstance(fields, dict):
                raise ValueError('Parameters must be a JSON object')
            if 'abc' in fields and len(str(fields['abc']).encode('utf-8')) > 1024 * 1024:
                raise ValueError('ABC score must be 1 MiB or smaller')
            task = str(fields.get('task', REMOTE_ALLOWED_TASKS[0])).lower()
            if task not in REMOTE_ALLOWED_TASKS:
                raise ValueError('Unsupported task: ' + task)
            if len(files) > REMOTE_MAX_IMAGES:
                raise ValueError(f'Only {REMOTE_MAX_IMAGES} input images are supported')
            if REMOTE_KIND in ('music', 'image_text') and files:
                raise ValueError('This model does not accept reference images')
            if REMOTE_KIND == 'video':
                minimum, maximum = {'t2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2),
                                    'r2v': (1, REMOTE_MAX_IMAGES)}[task] if task == 'r2v' else {
                                    't2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError('Input image count does not match the selected video task')
            if REMOTE_KIND == 'image_multi':
                minimum, maximum = {'text': (0, 0), 'edit': (1, 1), 'multi': (2, REMOTE_MAX_IMAGES)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError(f'Task {task} requires {minimum}–{maximum} images')
            for key in fields:
                if key not in REMOTE_FIELDS:
                    raise ValueError('Unsupported parameter: ' + key)
            remote_validate_fields(task, dict(fields, _image_count=len(files)))
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                self.send_json(409, {'error': inference_message})
                return
            for filename, data in files:
                if not isinstance(data, bytes) or len(data) > 20 * 1024 * 1024:
                    raise ValueError('Each image must be 20 MiB or smaller')
                if Path(filename).suffix.lower() not in {'.png', '.jpg', '.jpeg', '.webp'}:
                    raise ValueError('Only PNG/JPEG/WebP images are supported')
                from PIL import Image
                with Image.open(io.BytesIO(data)) as photo:
                    photo.verify()
            staging = Path(tempfile.mkdtemp(prefix='pending_', dir=REMOTE_ROOT))
            image_names = []
            for index, (filename, data) in enumerate(files, 1):
                target = staging / f'{index:03d}{Path(filename).suffix.lower()}'
                target.write_bytes(data)
                image_names.append(target.name)
            with REMOTE_JOB_LOCK:
                existing = REMOTE_JOBS.get(job_id)
                full = REMOTE_QUEUE.full()
                if not existing and not full:
                    folder = REMOTE_ROOT / job_id
                    staging.rename(folder)
                    staging = None
                    image_paths = [str(folder / name) for name in image_names]
                    REMOTE_JOBS[job_id] = {'id': job_id, 'task': task, 'status': 'queued',
                                           'created_at': time.time()}
                    REMOTE_QUEUE.put_nowait((job_id, fields, image_paths))
                    position = REMOTE_QUEUE.qsize()
            if existing:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'reused': True})
            elif full:
                self.send_json(429, {'error': 'Queue is full; retry later'}, 15)
            else:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'queue_position': position})
        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as error:
            self.send_json(400, {'error': str(error)})
        except (TimeoutError, ConnectionError, OSError) as error:
            self.send_json(503, {'error': f'Upload temporarily failed: {type(error).__name__}'}, 10)
        except Exception as error:
            self.send_json(500, {'error': type(error).__name__ + ': ' + str(error)[:250]})
        finally:
            if staging is not None:
                shutil.rmtree(staging, ignore_errors=True)
            REMOTE_UPLOAD_SLOTS.release()

'''
Convert numeric HTTP form fields into explicitly supported generator settings.
只轉換白名單中的 HTTP 參數，不執行客戶端提供的任何程式碼。
'''
def remote_generate(job_id, fields, image_paths):
    with REMOTE_ENGINE_LOCK:
        with REMOTE_JOB_LOCK:
            REMOTE_JOBS[job_id].update(status='running', _start_clock=time.monotonic())
        namespace = globals()
        keys = ['TASK', 'PROMPT', 'WIDTH', 'HEIGHT', 'SEED', 'LM_SEED', 'SECONDS', 'STEPS', 'CFG',
                'LYRICS', 'TAGS', 'STYLE', 'DURATION', 'GPU_ID', 'COT', 'ABC_PATH',
                'OUTPUT_NAME', 'REFERENCE_PATHS', 'ACTIVE_JOB', 'TASK_CONFIGS',
                'R2V_REF_DIR', 'NEGATIVE_PROMPT', 'SAMPLER', 'REFERENCE_SIZE',
                'OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT', 'RUN_SEED']
        before = {key: copy.deepcopy(namespace.get(key)) for key in keys if key in namespace}
        missing = set(keys) - set(before)
        try:
            task = fields.get('task', REMOTE_ALLOWED_TASKS[0]).lower()
            numerics = {'seed': int, 'lm_seed': int, 'width': int, 'height': int, 'steps': int,
                        'seconds': float, 'cfg': float, 'duration': int,
                        'image_strength': float, 'reference_size': int, 'audio_shift': float}
            inputs = {}
            for key, value in fields.items():
                inputs[key] = numerics[key](value) if key in numerics else value

            '''
            Resolve -1 once on the server so the exact value can be reproduced.
            在服務端一次決定 -1 的實際種子，方便重現結果。
            '''
            for seed_key in ('seed', 'lm_seed'):
                if seed_key in inputs:
                    if not -1 <= inputs[seed_key] < 2**32:
                        raise ValueError(f'{seed_key} must be -1 or between 0 and 4294967295')
                    if inputs[seed_key] == -1:
                        inputs[seed_key] = secrets.randbelow(2**32)
            with REMOTE_JOB_LOCK:
                for seed_key in ('seed', 'lm_seed'):
                    if seed_key in inputs:
                        REMOTE_JOBS[job_id][seed_key + '_used'] = inputs[seed_key]
            for dimension in ('width', 'height'):
                if dimension in inputs and not 0 < inputs[dimension] <= 4096:
                    raise ValueError('Width and height must be between 1 and 4096')
            for key in ('seconds', 'duration'):
                if key in inputs and not 0 < inputs[key] <= 3600:
                    raise ValueError('Duration must be between 0 and 3600 seconds')
            if 'reference_size' in inputs and not 64 <= inputs['reference_size'] <= 2048:
                raise ValueError('Reference size must be between 64 and 2048')
            if 'ref_image_size' in inputs and inputs['ref_image_size'] not in ('match', 'max'):
                raise ValueError('Reference image mode must be match or max')
            if 'image_strength' in inputs and not 0 <= inputs['image_strength'] <= 1:
                raise ValueError('Image strength must be between 0 and 1')
            if 'steps' in inputs and not 1 <= inputs['steps'] <= 200:
                raise ValueError('Steps must be between 1 and 200')
            if REMOTE_KIND == 'video':
                if 'TASK_CONFIGS' in namespace:
                    if task not in namespace['TASK_CONFIGS']:
                        raise ValueError('Run this task configuration cell before API generation')
                    updated = copy.deepcopy(namespace['TASK_CONFIGS'][task])
                    for field in ('prompt', 'width', 'height', 'seconds', 'steps', 'seed',
                                  'audio_shift', 'ref_image_size'):
                        if field in inputs:
                            updated[field] = inputs[field]
                    updated['output_name'] = f'remote_{job_id[:12]}.mp4'
                    if task == 'i2v':
                        updated['first_image'] = image_paths[0]
                    elif task == 'flf2v':
                        updated['first_image'], updated['last_image'] = image_paths
                    elif task == 'r2v':
                        updated['ref_images'] = image_paths
                        namespace['R2V_REF_DIR'] = Path(image_paths[0]).parent
                    namespace['TASK_CONFIGS'][task] = updated
                    namespace['TASK'] = task
                else:
                    namespace['ACTIVE_JOB'] = {'mode': task, 'prompt': inputs.get('prompt', namespace.get('PROMPT', '')),
                                               'run_name': f'remote_{job_id[:12]}',
                                               'first_path': image_paths[0] if image_paths else None,
                                               'last_path': image_paths[1] if len(image_paths) > 1 else None}
                    for key in ('width', 'height', 'seconds', 'seed', 'image_strength'):
                        if key in inputs:
                            namespace[key.upper()] = inputs[key]
            else:
                for key in ('prompt', 'width', 'height', 'seed', 'lm_seed', 'steps', 'cfg',
                            'lyrics', 'tags', 'style', 'duration', 'negative_prompt',
                            'sampler', 'reference_size', 'cot'):
                    if key in inputs:
                        namespace[key.upper()] = inputs[key]
                if REMOTE_KIND == 'image_multi':
                    namespace['TASK'] = task
                    namespace['REFERENCE_PATHS'] = image_paths
                if REMOTE_KIND == 'music':
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.mp3'
                    if 'ABC_PATH' in namespace:
                        namespace['ABC_PATH'] = ''
                    if 'abc' in inputs:
                        score = (REMOTE_ROOT / job_id / 'input.abc')
                        score.write_text(str(inputs['abc']), encoding='utf-8')
                        namespace['ABC_PATH'] = str(score)
                if REMOTE_KIND in ('image_multi', 'image_text'):
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.png'
            for output_key in ('OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT'):
                namespace.pop(output_key, None)
            gpu_ready, gpu_message, _ = remote_gpu_status()
            if not gpu_ready:
                raise RuntimeError(gpu_message)
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                raise RuntimeError(inference_message)
            exec(compile(REMOTE_SCRIPT, '<notebook-generation-cell>', 'exec'), namespace)
            paths = []
            if REMOTE_KIND == 'video':
                paths = [namespace.get('LAST_OUTPUT') or namespace.get('OUTPUT_MP4')]
            elif REMOTE_KIND == 'image_text':
                paths = [namespace.get('OUTPUT_PATH')]
            elif REMOTE_KIND == 'image_multi':
                paths = [namespace.get('output')]
            elif REMOTE_KIND == 'music':
                paths = namespace.get('OUTPUTS', []) if 'OUTPUTS' in namespace else [namespace.get('output')]
            results = [Path(path) for path in paths if path is not None]
            if REMOTE_KIND == 'music' and REMOTE_SERVICE_ID.startswith('06_YuE2_'):
                results += [sidecar for path in list(results)
                            for ext in ('.json', '.abc')
                            if (sidecar := path.with_suffix(ext)).is_file()]
            if not results or any(not path.is_file() for path in results):
                raise FileNotFoundError('Generation finished without a saved result file')
            folder = REMOTE_ROOT / job_id
            if len(results) == 1:
                target = folder / f'result{results[0].suffix.lower()}'
                shutil.copy2(results[0], target)
            else:
                target = folder / 'results.zip'
                with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_STORED) as archive:
                    for result in results:
                        archive.write(result, arcname=result.name)
            last_frame = None
            if REMOTE_KIND == 'video' and shutil.which('ffprobe') and shutil.which('ffmpeg'):
                try:
                    probe = subprocess.run(['ffprobe', '-v', 'error', '-count_frames',
                                            '-select_streams', 'v:0', '-show_entries',
                                            'stream=nb_read_frames', '-of', 'json', str(target)],
                                           check=True, text=True, capture_output=True, timeout=120)
                    index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1
                    if index >= 0:
                        last_frame = folder / 'last_frame.png'
                        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(target),
                                        '-vf', rf'select=eq(n\,{index})', '-frames:v', '1',
                                        str(last_frame)], check=True)
                except (OSError, ValueError, KeyError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
                    last_frame = None

            '''
            Remove only source files bearing this remote job's unique prefix.
            僅清理本次遠端工作專屬原始輸出，保留 Notebook 手動生成的檔案。
            '''
            for original in results:
                if original.name.startswith(f'remote_{job_id[:12]}') and original != target:
                    try:
                        original.unlink(missing_ok=True)
                    except OSError:
                        pass
            comfy = globals().get('COMFY')
            if comfy is not None and (Path(comfy) / 'output').exists():
                for artifact in (Path(comfy) / 'output').rglob(f'remote_{job_id[:12]}*'):
                    if artifact.is_file():
                        try:
                            artifact.unlink()
                        except OSError:
                            pass
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='done', filename=target.name,
                                           bytes=target.stat().st_size, _path=str(target),
                                           last_frame_available=bool(last_frame),
                                           _last_frame=str(last_frame) if last_frame else None)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            for key in missing:
                namespace.pop(key, None)
            namespace.update(before)

'''
Use one GPU worker, regardless of the number of HTTP request threads.
不論有多少 HTTP 連線，都只有一個 GPU worker 消費佇列。
'''
def remote_worker():
    while True:
        job_id, fields, image_paths = REMOTE_QUEUE.get()
        try:
            remote_generate(job_id, fields, image_paths)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id]['finished_at'] = time.time()
            try:
                remote_remove_inputs(job_id)
            except OSError as error:
                print('API input cleanup warning:', str(error)[:180])
            finally:
                REMOTE_QUEUE.task_done()

if not (globals().get('REMOTE_WORKER') and REMOTE_WORKER.is_alive()):
    REMOTE_WORKER = threading.Thread(target=remote_worker, daemon=True)
    REMOTE_WORKER.start()

if not (globals().get('REMOTE_MAINTENANCE') and REMOTE_MAINTENANCE.is_alive()):
    REMOTE_MAINTENANCE = threading.Thread(target=remote_maintenance, daemon=True)
    REMOTE_MAINTENANCE.start()

old_tunnel = globals().get('REMOTE_NGROK_PROC')
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()
        old_tunnel.wait(timeout=5)

NGROK_URL = ''

'''
Install the official ngrok agent only if no usable binary exists.
系統尚無 ngrok 才下載官方執行檔，不重複安裝。
'''
ngrok_bin = shutil.which('ngrok') or str(ROOT / 'ngrok')
if not Path(ngrok_bin).is_file():
    url = 'https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz'
    with urllib.request.urlopen(url, timeout=90) as response:
        blob = response.read()
    with tarfile.open(fileobj=io.BytesIO(blob), mode='r:gz') as archive:
        entry = next((member for member in archive.getmembers() if Path(member.name).name == 'ngrok' and member.isfile()), None)
        if entry is None:
            raise RuntimeError('ngrok 官方下載檔案缺少執行檔')
        Path(ngrok_bin).write_bytes(archive.extractfile(entry).read())
    Path(ngrok_bin).chmod(0o755)

ngrok_token = getpass('輸入 ngrok Authtoken：').strip()
if not ngrok_token:
    raise ValueError('未輸入 ngrok Authtoken')

result = subprocess.run([ngrok_bin, 'config', 'add-authtoken', ngrok_token], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok token 設定失敗（請確認 Token 是否有效）')

del ngrok_token
result = subprocess.run([ngrok_bin, 'config', 'check'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok 設定格式檢查失敗：' + result.stderr[-500:])

server = globals().get('REMOTE_SERVER')
server_thread = globals().get('REMOTE_SERVER_THREAD')
server_is_current = (
    server is not None
    and server_thread is not None
    and server_thread.is_alive()
    and getattr(server, 'notebook_id', None) == REMOTE_SERVICE_ID
)
if server is not None and not server_is_current:
    if server_thread is not None and server_thread.is_alive():
        server.shutdown()
    server.server_close()

if not server_is_current:
    try:
        REMOTE_SERVER = ThreadingHTTPServer(('127.0.0.1', REMOTE_PORT), RemoteHandler)
    except OSError as error:
        raise RuntimeError(f'本機 API Port {REMOTE_PORT} 已被其他服務占用') from error
    REMOTE_SERVER.notebook_id = REMOTE_SERVICE_ID
    REMOTE_SERVER.daemon_threads = True
    REMOTE_SERVER_THREAD = threading.Thread(target=REMOTE_SERVER.serve_forever, daemon=True)
    REMOTE_SERVER_THREAD.start()
    print('已啟動新的本機 API Server。')
else:
    REMOTE_SERVER.RequestHandlerClass = RemoteHandler
    print('已沿用本機 API Server；現有生成工作與結果保持不變。')

with urllib.request.urlopen(f'http://127.0.0.1:{REMOTE_PORT}/health', timeout=4) as response:
    if json.load(response).get('ok') is not True:
        raise RuntimeError('本機 API Server 尚未就緒')

'''
Use the agent inspection API to obtain the actual public URL.
透過 ngrok 本機檢查 API 取得實際對外 HTTPS 網址。
'''
NGROK_LOG = REMOTE_ROOT / 'ngrok.log'
with NGROK_LOG.open('w') as log:
    REMOTE_NGROK_PROC = subprocess.Popen([ngrok_bin, 'http', str(REMOTE_PORT)], stdout=log,
                                         stderr=subprocess.STDOUT, start_new_session=True)

NGROK_URL = ''
for attempt in range(25):
    if REMOTE_NGROK_PROC.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=2) as response:
            tunnels = json.load(response).get('tunnels', [])
        NGROK_URL = next((item['public_url'] for item in tunnels
                          if item.get('public_url', '').startswith('https://') and
                          str(REMOTE_PORT) in item.get('config', {}).get('addr', '')), '')
        if NGROK_URL:
            break
    except (OSError, ValueError, KeyError):
        pass
    time.sleep(1)

if not NGROK_URL:
    raise RuntimeError('ngrok URL 建立失敗：\n' + NGROK_LOG.read_text(errors='replace')[-2500:])

print('ngrok HTTPS URL:', NGROK_URL)
print('提醒：公開的 ngrok API URL 不需要額外驗證，請只分享給授課學生。')
print('GET /health | GET /schema | POST /generate | GET /jobs/{id} | GET /jobs/{id}/file')
print('影片輸出另提供：GET /jobs/{id}/last-frame')
print('✓ 第 7 節完成：選用 FIFO／ngrok API 已啟動。')


8. （選用）遠端 API 與 client 使用說明


In [ ]:
'''
Query the active bridge for supported parameters and task names.
從正在執行的橋接服務讀取實際支援的參數與模式。
'''
import json
import urllib.request
request = urllib.request.Request(f'http://127.0.0.1:{REMOTE_PORT}/schema')
with urllib.request.urlopen(request, timeout=5) as response:
    schema = json.load(response)

print(json.dumps(schema, ensure_ascii=False, indent=2))
print('用法：POST NGROK_URL + "/generate"（JSON 或 multipart/images），取得 job_id；')
print('輪詢 GET /jobs/{job_id}；完成後下載 GET /jobs/{job_id}/file。')
print('本機 requests 範例：06_Z-Image-Turbo_圖片生成_client.py（與本 Notebook 同名對應）。')
print('使用前只需在對應的 client.py 填入 API_URL。')
print('本服務使用單一 FIFO worker，遠端請求不會同時啟動多筆 GPU 推論。')
print('✓ 第 8 節完成：API Schema 和 client 說明已顯示。')
